In [1]:
# ============================================================
# 03.5_stage_classifier_combination.ipynb
# Hierarchical 4-class classifier (stage1~stage4)
# Node label conventions (MUST match training):
#   m1: y=1 => stage4,   y=0 => stage1/2/3      (123 vs 4)
#   m2: y=0 => stage1,   y=1 => stage2/3        (1 vs 23)
#   m3: y=0 => stage2,   y=1 => stage3          (2 vs 3)
# ============================================================

In [2]:

import os
import numpy as np
from pathlib import Path
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt


In [3]:
import numpy as np
import torch
import torch.nn.functional as F

class M2Stacker:
    """
    Stacking ensemble for m2: (stage1) vs (stage2,3)
    - base models: PyTorch binary classifiers output logits (B,2)
    - meta_clf: sklearn model with predict_proba(X)
    Feature design:
      X_meta = concat([p0, p1] for each base model) => shape (B, 2*K)
      where p0 = P(class0=stage1), p1 = P(class1=stage23)
    Output:
      p_stage1 (tensor, shape (B,))
    """
    def __init__(self, models, meta_clf, device="cuda"):
        self.models = models
        self.meta = meta_clf
        self.device = device

    @torch.no_grad()
    def _make_meta_features(self, x):
        """
        x: (B,3,H,W) torch tensor
        return: numpy array (B, 2*K)
        """
        x = x.to(self.device)

        feats = []
        for m in self.models:
            logits = m(x)                 # (B,2)
            p = F.softmax(logits, dim=1)  # (B,2)
            feats.append(p)

        feats = torch.cat(feats, dim=1)   # (B, 2*K)
        return feats.detach().cpu().numpy()

    @torch.no_grad()
    def p_stage1(self, x):
        """
        return p(stage1) for each sample in batch
        x: (B,3,H,W)
        """
        X_meta = self._make_meta_features(x)           # (B,2*K)
        prob_stage1 = self.meta.predict_proba(X_meta)[:, 0]  # class0 = stage1
        return torch.tensor(prob_stage1, dtype=torch.float32, device="cpu")


In [4]:
# ----------------------------
# 0) Config
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

PROJ = Path(r"D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理").resolve()
DATA_ROOT = PROJ / "stage_cls_dataset"   # 4-class folder: stage_1..stage_4

# --- IMPORTANT: set correct ckpt paths ---
M1_CKPT = PROJ / "outputs_bin(1,2,3),(4)" / "_best_model" / "best_densenet121.pth"
CFG_PATH = PROJ / "03.5_combination" / "03.25_m2_stacking_top3" / "config.json"# !!! m3 must be trained on (2) vs (3), NOT (1,2) vs (3,4)
M3_CKPT = PROJ / "outputs_bin(2),(3)" / "_best_model" / "best_densenet121.pth"

# output folder for this combination result
OUT_DIR = PROJ / "outputs_hierarchical_4class"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# thresholds per node (you can tune later)
THR1 = 0.5  # for m1: stage4?
# THR2 = 0.5  # for m2: stage1?
THR3 = 0.5  # for m3: stage3?



device = cuda


In [5]:
# ----------------------------
# 1) Build same transforms as your val/test (no flip)
# ----------------------------
val_tf = T.Compose([
    T.Resize((384, 384)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

# 4-class dataset (NO wrapper here!)
base_ds = ImageFolder(DATA_ROOT, transform=val_tf)
print("Total images:", len(base_ds))
print("class_to_idx:", base_ds.class_to_idx)

# If you want to reuse the SAME split indices as before:
# (Recommended) load your saved indices if you have them.
# Otherwise we do a deterministic stratified split again (same logic as your old code).

train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15
g = torch.Generator().manual_seed(42)

labels4 = torch.tensor([y for _, y in base_ds.samples])  # 0..3
num_classes4 = len(base_ds.classes)

train_idx_all, val_idx_all, test_idx_all = [], [], []
for c in range(num_classes4):
    idx_c = torch.where(labels4 == c)[0]
    idx_c = idx_c[torch.randperm(len(idx_c), generator=g)]
    n = len(idx_c)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)
    train_idx_all.append(idx_c[:n_train])
    val_idx_all.append(idx_c[n_train:n_train+n_val])
    test_idx_all.append(idx_c[n_train+n_val:])

train_idx = torch.cat(train_idx_all)
val_idx   = torch.cat(val_idx_all)
test_idx  = torch.cat(test_idx_all)

# shuffle indices
train_idx = train_idx[torch.randperm(len(train_idx), generator=g)]
val_idx   = val_idx[torch.randperm(len(val_idx), generator=g)]
test_idx  = test_idx[torch.randperm(len(test_idx), generator=g)]

print("Split sizes:", len(train_idx), len(val_idx), len(test_idx))

val_set  = Subset(base_ds, val_idx.tolist())
test_set = Subset(base_ds, test_idx.tolist())

val_loader_4 = DataLoader(val_set, batch_size=16, shuffle=False)
test_loader_4 = DataLoader(test_set, batch_size=16, shuffle=False)

Total images: 287
class_to_idx: {'stage_1': 0, 'stage_2': 1, 'stage_3': 2, 'stage_4': 3}
Split sizes: 200 41 46


In [6]:
# ============================================================
# MUST HAVE: create_model() (same as training)
# Put this cell BEFORE you create/load m1/m2/m3
# ============================================================

import torch
import torch.nn as nn
import torchvision.models as models

def create_model(model_name: str, num_classes: int = 2):
    """
    Build a torchvision model backbone with pretrained weights,
    and replace the final classification layer to num_classes.

    model_name options (match your training notebooks):
      efficientnet_b0, efficientnet_b1, resnet50,
      convnext_tiny, convnext_small, convnext_base,
      vit_b_16, densenet121, densenet169
    """
    model_name = model_name.lower()

    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "efficientnet_b1":
        weights = models.EfficientNet_B1_Weights.DEFAULT
        model = models.efficientnet_b1(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        model = models.convnext_tiny(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_small":
        weights = models.ConvNeXt_Small_Weights.DEFAULT
        model = models.convnext_small(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_base":
        weights = models.ConvNeXt_Base_Weights.DEFAULT
        model = models.convnext_base(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "vit_b_16":
        weights = models.ViT_B_16_Weights.DEFAULT
        model = models.vit_b_16(weights=weights)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)

    elif model_name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT
        model = models.densenet121(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    elif model_name == "densenet169":
        weights = models.DenseNet169_Weights.DEFAULT
        model = models.densenet169(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return model


In [7]:
# ----------------------------
# 2) Create models (must match training code)
#    You must already have create_model(model_name, num_classes) defined in this notebook.
# ----------------------------
# Example:
# m1 = create_model("densenet121", num_classes=2).to(device)
# ...

import json, joblib

# ----------------------------
# 2) Create models (m1/m3 are single .pth, m2 is stacking)
# ----------------------------
m1 = create_model("densenet121", num_classes=2).to(device)
m3 = create_model("densenet121", num_classes=2).to(device)

# load weights (m1/m3)
m1.load_state_dict(torch.load(M1_CKPT, map_location=device))
m3.load_state_dict(torch.load(M3_CKPT, map_location=device))
m1.eval(); m3.eval()

# ----------------------------
# 2.1) Load m2 stacking (meta .pkl + top3 base .pth)
# ----------------------------
cfg = json.loads(Path(CFG_PATH).read_text(encoding="utf-8"))

# load meta classifier (.pkl)
meta_clf = joblib.load(cfg["meta_clf_path"])

# load base models (.pth)
m2_models = []
for item in cfg["m2_base_models"]:
    name = item["model_name"]
    ckpt = Path(item["ckpt_in_comb_dir"])   # 通常是已經 copy 到 combination 資料夾內的路徑
    m = create_model(name, num_classes=2).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    m2_models.append(m)

m2 = M2Stacker(m2_models, meta_clf, device=device)

# thresholds: THR2 can come from config
THR2 = float(cfg.get("thr2_default", 0.5))

print("✅ Loaded m1/m2(stacking)/m3 checkpoints.")
print("✅ THR2 =", THR2)



✅ Loaded m1/m2(stacking)/m3 checkpoints.
✅ THR2 = 0.5


In [8]:
# ----------------------------
# 3) Hierarchical prediction
# ----------------------------
@torch.no_grad()
def predict_hierarchical(x, m1, m2, m3, device="cuda", thr1=0.5, thr2=0.5, thr3=0.5):
    """
    x: [1,3,H,W] tensor
    return: stage_id in {0,1,2,3} => stage_1..stage_4
    """
    x = x.to(device)

    # Node-1: 123 vs 4 (y=1 means stage4)
    p_stage4 = F.softmax(m1(x), dim=1)[0, 1].item()
    if p_stage4 >= thr1:
        return 3  # stage_4

    # Node-2: 1 vs 23 (y=0 means stage1)
    p_stage1 = m2.p_stage1(x)[0].item()
    if p_stage1 >= thr2:
        return 0  # stage_1

    # Node-3: 2 vs 3 (y=1 means stage3)
    p_stage3 = F.softmax(m3(x), dim=1)[0, 1].item()
    if p_stage3 >= thr3:
        return 2  # stage_3
    else:
        return 1  # stage_2


In [9]:
# from sklearn.metrics import accuracy_score, f1_score

# def eval_hierarchical_quick(loader, m1, m2, m3, device="cuda", thr1=0.5, thr2=0.5, thr3=0.5):
#     y_true_all, y_pred_all = [], []

#     for imgs, labels4 in loader:
#         imgs = imgs.to(device, non_blocking=True)
#         # 一張一張走樹（簡單但慢；你的資料量不大可以接受）
#         for i in range(imgs.size(0)):
#             pred4 = predict_hierarchical(imgs[i:i+1], m1, m2, m3, device=device, thr1=thr1, thr2=thr2, thr3=thr3)
#             y_pred_all.append(pred4)
#         y_true_all.extend(labels4.cpu().numpy().tolist())

#     acc = accuracy_score(y_true_all, y_pred_all)
#     macro = f1_score(y_true_all, y_pred_all, average="macro")
#     return acc, macro


In [10]:
# # =========================
# # Thr2 grid search (only)
# # =========================
# thr2_list = np.arange(0.20, 0.61, 0.01)  # 建議先小範圍密集搜

# best_t2 = None
# best_macro = -1
# best_acc = -1

# for t2 in thr2_list:
#     acc, macro = eval_hierarchical_quick(
#         val_loader_4, m1, m2, m3,
#         device=device,
#         thr1=THR1,      # 固定
#         thr2=float(t2), # 只調這個
#         thr3=THR3       # 固定
#     )
#     if macro > best_macro:
#         best_macro = macro
#         best_acc = acc
#         best_t2 = float(t2)
#         print(f"New best thr2={best_t2:.2f} | val_macro_f1={best_macro:.4f}, val_acc={best_acc:.4f}")

# print("\n✅ BEST thr2 =", best_t2)

# # 套用到後面的正式 eval
# THR2 = best_t2
# print("✅ Set THR2 =", THR2)



In [11]:
# ----------------------------
# 4) Evaluation on 4-class loader
# ----------------------------
def eval_hierarchical(loader, m1, m2, m3, device="cuda", thr1=0.5, thr2=0.5, thr3=0.5, class_names=None):
    y_true_all, y_pred_all = [], []

    for imgs, labels4 in loader:
        for i in range(imgs.size(0)):
            x = imgs[i:i+1]
            pred4 = predict_hierarchical(x, m1, m2, m3, device=device, thr1=thr1, thr2=thr2, thr3=thr3)
            y_pred_all.append(pred4)

        y_true_all.extend(labels4.cpu().numpy().tolist())

    acc = accuracy_score(y_true_all, y_pred_all)
    macro = f1_score(y_true_all, y_pred_all, average="macro")
    cm = confusion_matrix(y_true_all, y_pred_all)

    print("Hierarchical ACC:", acc)
    print("Hierarchical macro_f1:", macro)
    print("Confusion matrix (4x4):\n", cm)

    if class_names is not None:
        print("\nReport:\n", classification_report(y_true_all, y_pred_all, target_names=class_names, digits=4))
    else:
        print("\nReport:\n", classification_report(y_true_all, y_pred_all, digits=4))

    return {"acc": acc, "macro_f1": macro, "cm": cm, "y_true": y_true_all, "y_pred": y_pred_all}


def save_cm_png(cm, class_names, save_path: Path, title="Hierarchical"):
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, cm[i, j],
                    ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

In [12]:

# ----------------------------
# 5) Run eval + save artifacts
# ----------------------------
class_names4 = ["stage_1", "stage_2", "stage_3", "stage_4"]

print("\n=== VAL (Hierarchical) ===")
val_res = eval_hierarchical(val_loader_4, m1, m2, m3, device=device, thr1=THR1, thr2=THR2, thr3=THR3, class_names=class_names4)
save_cm_png(val_res["cm"], class_names4, OUT_DIR / "confusion_matrix_val.png", title="Hierarchical | val")

print("\n=== TEST (Hierarchical) ===")
test_res = eval_hierarchical(test_loader_4, m1, m2, m3, device=device, thr1=THR1, thr2=THR2, thr3=THR3, class_names=class_names4)
save_cm_png(test_res["cm"], class_names4, OUT_DIR / "confusion_matrix_test.png", title="Hierarchical | test")

# Save a brief summary
(OUT_DIR / "SUMMARY.txt").write_text(
    f"Hierarchical 4-class results\n"
    f"thr1={THR1}, thr2={THR2}, thr3={THR3}\n\n"
    f"[VAL]  acc={val_res['acc']:.6f}, macro_f1={val_res['macro_f1']:.6f}\n"
    f"[TEST] acc={test_res['acc']:.6f}, macro_f1={test_res['macro_f1']:.6f}\n",
    encoding="utf-8"
)

print("\n✅ Saved hierarchical results to:", OUT_DIR)


=== VAL (Hierarchical) ===
Hierarchical ACC: 0.9512195121951219
Hierarchical macro_f1: 0.9454156954156954
Confusion matrix (4x4):
 [[ 6  0  1  0]
 [ 0 16  0  0]
 [ 0  1  8  0]
 [ 0  0  0  9]]

Report:
               precision    recall  f1-score   support

     stage_1     1.0000    0.8571    0.9231         7
     stage_2     0.9412    1.0000    0.9697        16
     stage_3     0.8889    0.8889    0.8889         9
     stage_4     1.0000    1.0000    1.0000         9

    accuracy                         0.9512        41
   macro avg     0.9575    0.9365    0.9454        41
weighted avg     0.9527    0.9512    0.9507        41


=== TEST (Hierarchical) ===
Hierarchical ACC: 0.8913043478260869
Hierarchical macro_f1: 0.8765182186234818
Confusion matrix (4x4):
 [[ 5  2  1  0]
 [ 0 17  1  0]
 [ 0  1  8  0]
 [ 0  0  0 11]]

Report:
               precision    recall  f1-score   support

     stage_1     1.0000    0.6250    0.7692         8
     stage_2     0.8500    0.9444    0.8947      